<a href="https://colab.research.google.com/github/hadri1-bisch/SCM/blob/main/scm_survey_part_3_v3_changeofprice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Modélisation du questionnaire:choix entre un pull neuf et un pull de seconde main**

NB : le produit choisi est un pull. Les prix ont été adaptés à ce produit :
- Neuf : 40, 60, 100 CHF (fast fashion, marque moyenne, bonne marque)
- Seconde main : 20, 40 CHF (particulier, magasin professionnel)

**A) créations des profils**

In [ ]:
import itertools
import pandas as pd
import numpy as np


In [ ]:
# Définition des attributs PAR ALTERNATIVE (design labellisé : Neuf vs Seconde main)

# ---- Alternative "Neuf" : 3 attributs ----
prix_neuf_niveaux = [40, 60, 100]
assortiment_neuf_niveaux = [1, 2]  # 1: Limité, 2: Extensif
garantie_neuf_niveaux = [1, 2]     # 1: Sans retour, 2: Retours 30j

profils_neuf = list(itertools.product(
    prix_neuf_niveaux, assortiment_neuf_niveaux, garantie_neuf_niveaux
))
df_profils_neuf = pd.DataFrame(
    profils_neuf, columns=['Prix', 'Assortiment', 'Garantie']
)

# ---- Alternative "Seconde main" : 4 attributs (Hygiène est spécifique à la SM) ----
prix_sm_niveaux = [20, 40]
hygiene_sm_niveaux = [1, 2]        # 1: Standard (particulier), 2: Certifié professionnel
assortiment_sm_niveaux = [1, 2]    # 1: Limité, 2: Extensif
garantie_sm_niveaux = [1, 2]       # 1: Sans retour, 2: Retours 30j

profils_sm = list(itertools.product(
    prix_sm_niveaux, hygiene_sm_niveaux, assortiment_sm_niveaux, garantie_sm_niveaux
))
df_profils_sm = pd.DataFrame(
    profils_sm, columns=['Prix', 'Hygiene', 'Assortiment', 'Garantie']
)

print(f"Profils Neuf (plein factoriel) : {len(df_profils_neuf)}")
print(df_profils_neuf.to_string(index=False))
print()
print(f"Profils Seconde main (plein factoriel) : {len(df_profils_sm)}")
print(df_profils_sm.to_string(index=False))

Profils Neuf (plein factoriel) : 12
 Prix  Assortiment  Garantie
   40            1         1
   40            1         2
   40            2         1
   40            2         2
   60            1         1
   60            1         2
   60            2         1
   60            2         2
  100            1         1
  100            1         2
  100            2         1
  100            2         2

Profils Seconde main (plein factoriel) : 16
 Prix  Hygiene  Assortiment  Garantie
   20        1            1         1
   20        1            1         2
   20        1            2         1
   20        1            2         2
   20        2            1         1
   20        2            1         2
   20        2            2         1
   20        2            2         2
   40        1            1         1
   40        1            1         2
   40        1            2         1
   40        1            2         2
   40        2            1         1
   40     

B) **Filtrage des profils iréalsites**

Neuf : 3 attributs
- Prix : 40, 60, 100 CHF
- Assortiment : 1 (Limité), 2 (Extensif)
- Garantie : 1 (Sans retour), 2 (Retours 30j)

Seconde main : 4 attributs
- Prix : 20, 40 CHF
- Hygiène : 1 (Standard / particulier), 2 (Premium / certifié professionnel)
- Assortiment : 1 (Limité), 2 (Extensif) — extensif possible uniquement si Hygiène = Premium
- Garantie : 1 (Sans retour), 2 (Retours 30j)

In [ ]:
# Filtrage des profils Seconde main selon la contrainte conditionnelle
def filtre_profils_sm(row):
    # Hygiène standard (particulier) -> pièce unique -> assortiment limité uniquement
    if row['Hygiene'] == 1 and row['Assortiment'] == 2:
        return False
    return True

df_profils_sm_valides = (
    df_profils_sm[df_profils_sm.apply(filtre_profils_sm, axis=1)]
    .reset_index(drop=True)
)

# Aucune contrainte de réalisme identifiée pour le neuf -> tous les profils sont gardés
df_profils_neuf_valides = df_profils_neuf.reset_index(drop=True)

print(f"Profils Neuf valides : {len(df_profils_neuf_valides)}")
print(df_profils_neuf_valides.to_string(index=False))
print()
print(f"Profils Seconde main valides : {len(df_profils_sm_valides)}")
print(df_profils_sm_valides.to_string(index=False))

assert len(df_profils_neuf_valides) == 12, "Le nombre de profils Neuf valides ne correspond pas à l'attendu"
assert len(df_profils_sm_valides) == 12, "Le nombre de profils Seconde main valides ne correspond pas à l'attendu"

Profils Neuf valides : 12
 Prix  Assortiment  Garantie
   40            1         1
   40            1         2
   40            2         1
   40            2         2
   60            1         1
   60            1         2
   60            2         1
   60            2         2
  100            1         1
  100            1         2
  100            2         1
  100            2         2

Profils Seconde main valides : 12
 Prix  Hygiene  Assortiment  Garantie
   20        1            1         1
   20        1            1         2
   20        2            1         1
   20        2            1         2
   20        2            2         1
   20        2            2         2
   40        1            1         1
   40        1            1         2
   40        2            1         1
   40        2            1         2
   40        2            2         1
   40        2            2         2


**C) Question a insérer dans le questionnaires**

créations de toutes les paires candidates:

In [ ]:
from itertools import product

In [ ]:
# Générer toutes les paires : seconde main vs neuf
paires = []

for (idx_sm, profil_sm), (idx_neuf, profil_neuf) in product(
    df_profils_sm_valides.iterrows(),
    df_profils_neuf_valides.iterrows()
):
    paires.append({
        "Prix_SM": profil_sm["Prix"],
        "Hygiene_SM": profil_sm["Hygiene"],
        "Tailles_SM": profil_sm["Assortiment"],
        "Retours_SM": profil_sm["Garantie"],
        "Prix_neuf": profil_neuf["Prix"],
        "Tailles_neuf": profil_neuf["Assortiment"],
        "Retours_neuf": profil_neuf["Garantie"]
    })

df_paires_candidates = pd.DataFrame(paires)

print(f"Nombre de paires candidates : {len(df_paires_candidates)}")
print(df_paires_candidates.to_string(index=False))

assert len(df_paires_candidates) == 144, "Le nombre de paires candidates ne correspond pas à l'attendu"

Nombre de paires candidates : 144
 Prix_SM  Hygiene_SM  Tailles_SM  Retours_SM  Prix_neuf  Tailles_neuf  Retours_neuf
      20           1           1           1         40             1             1
      20           1           1           1         40             1             2
      20           1           1           1         40             2             1
      20           1           1           1         40             2             2
      20           1           1           1         60             1             1
      20           1           1           1         60             1             2
      20           1           1           1         60             2             1
      20           1           1           1         60             2             2
      20           1           1           1        100             1             1
      20           1           1           1        100             1             2
      20           1           1          

In [ ]:
# Supprimer les questions évidentes :
# une option est au moins aussi bonne sur le prix, les tailles ET les retours
def est_evident(row):
    sm_mieux = (row["Prix_SM"] <= row["Prix_neuf"]
                and row["Tailles_SM"] >= row["Tailles_neuf"]
                and row["Retours_SM"] >= row["Retours_neuf"])
    neuf_mieux = (row["Prix_neuf"] <= row["Prix_SM"]
                  and row["Tailles_neuf"] >= row["Tailles_SM"]
                  and row["Retours_neuf"] >= row["Retours_SM"])
    return sm_mieux or neuf_mieux

df_paires_candidates = (
    df_paires_candidates[~df_paires_candidates.apply(est_evident, axis=1)]
    .reset_index(drop=True)
)
print("Paires restantes :", len(df_paires_candidates))

Paires restantes : 63


**Choix des questions**
but min d-error

In [ ]:
pip install choicedesign

In [ ]:
import choicedesign as cd

Design D-efficient avec priors (pull)
- Prix linéaire : un seul coefficient par franc
- Retours séparés : un effet pour la seconde main, un pour le neuf
- Priors supposés (signe attendu), à remplacer par les estimations du questionnaire test

In [ ]:
from choicedesign.design import EffDesign
from choicedesign.expressions import Attribute, Parameter, ASC

# --------------------------------------------------
# 1. Attributs (inchangé)
# --------------------------------------------------
prix_sm = Attribute("Prix_SM", [100, 150])
hygiene_sm = Attribute("Hygiene_SM", [1, 2])
tailles_sm = Attribute("Tailles_SM", [1, 2])
retours_sm = Attribute("Retours_SM", [1, 2])

prix_neuf = Attribute("Prix_neuf", [100, 150, 300])
tailles_neuf = Attribute("Tailles_neuf", [1, 2])
retours_neuf = Attribute("Retours_neuf", [1, 2])

# --------------------------------------------------
# 2. Paramètres
# Prix LINÉAIRE : un seul coefficient "par franc"
# (remplace beta_prix_150 et beta_prix_300)
# Priors supposés (signe attendu), pas tirés de la littérature
# --------------------------------------------------
beta_prix = Parameter("beta_prix", -0.03)               # effet de chaque franc
beta_tailles = Parameter("beta_tailles", 0.3)           # ≈ 10 CHF
beta_hygiene = Parameter("beta_hygiene", 0.5)           # ≈ 17 CHF
beta_retours_sm = Parameter("beta_retours_sm", 0.4)     # ≈ 13 CHF
beta_retours_neuf = Parameter("beta_retours_neuf", 0.2) # ≈ 7 CHF

asc_sm = ASC("asc_sm", 0.0)

# --------------------------------------------------
# 3. Fonctions d'utilité
# Le prix entre directement comme un nombre
# --------------------------------------------------
V_sm = (
    asc_sm
    + beta_prix * prix_sm
    + beta_hygiene * (hygiene_sm == 2)
    + beta_tailles * (tailles_sm == 2)
    + beta_retours_sm * (retours_sm == 2)
)

V_neuf = (
    beta_prix * prix_neuf
    + beta_tailles * (tailles_neuf == 2)
    + beta_retours_neuf * (retours_neuf == 2)
)

V = {
    1: V_sm,
    2: V_neuf
}

candidats:

In [ ]:
# --------------------------------------------------
# 4. Préparer les 144 paires candidates
# --------------------------------------------------

colonnes_modele = [
    "Prix_SM",
    "Hygiene_SM",
    "Tailles_SM",
    "Retours_SM",
    "Prix_neuf",
    "Tailles_neuf",
    "Retours_neuf"
]

candidates = df_paires_candidates.reset_index(drop=True).copy()


# --------------------------------------------------
# 5. Créer l'évaluateur D-efficient
# --------------------------------------------------

design = EffDesign(
    X=[
        prix_sm,
        hygiene_sm,
        tailles_sm,
        retours_sm,
        prix_neuf,
        tailles_neuf,
        retours_neuf
    ],
    ncs=12
)


def evaluer_selection(indices):
    plan = candidates.iloc[indices][colonnes_modele].copy()

    # au moins 4 questions avec hygiène standard
    if (plan["Hygiene_SM"] == 1).sum() < 4:
        return np.inf, 0.0

    plan.insert(0, "CS", range(1, len(plan) + 1))

    d_error, equilibre = design.evaluate(
        plan,
        V=V,
        criterion="d"
    )

    return float(d_error), float(equilibre)
# --------------------------------------------------
# 6. Recherche locale avec plusieurs départs aléatoires
#    (version limitée en temps : échantillonnage des candidats testés à chaque swap)
# --------------------------------------------------

rng = np.random.default_rng(42)

meilleur_plan = None
meilleur_d_error = np.inf
meilleur_equilibre = None

nombre_depart = 5
iterations_max = 25
taille_echantillon = 70

for depart in range(nombre_depart):

    # tirer un départ qui respecte déjà la règle des 4 "standard"
    while True:
        selection = list(
            rng.choice(len(candidates), size=12, replace=False)
        )
        if (candidates.iloc[selection]["Hygiene_SM"] == 1).sum() >= 4:
            break

    d_error, equilibre = evaluer_selection(selection)

    for iteration in range(iterations_max):

        indices_selectionnes = set(selection)
        candidats_restants = [
            i for i in range(len(candidates)) if i not in indices_selectionnes
        ]
        candidats_testes = rng.choice(
            candidats_restants,
            size=min(taille_echantillon, len(candidats_restants)),
            replace=False
        )

        meilleur_echange = None
        meilleur_d_local = d_error
        meilleur_eq_local = equilibre

        for position in range(12):

            for candidate_idx in candidats_testes:

                essai = selection.copy()
                essai[position] = candidate_idx

                d_essai, eq_essai = evaluer_selection(essai)

                if (
                    np.isfinite(d_essai)
                    and d_essai < meilleur_d_local - 1e-10
                ):
                    meilleur_d_local = d_essai
                    meilleur_eq_local = eq_essai
                    meilleur_echange = essai

        if meilleur_echange is None:
            break

        selection = meilleur_echange
        d_error = meilleur_d_local
        equilibre = meilleur_eq_local

    print(f"Départ {depart + 1}/{nombre_depart} terminé — D-error : {d_error:.4f}")

    if d_error < meilleur_d_error:
        meilleur_d_error = d_error
        meilleur_equilibre = equilibre
        meilleur_plan = selection.copy()

# --------------------------------------------------
# 7. Afficher les 12 questions sélectionnées
# --------------------------------------------------

df_questions = candidates.iloc[meilleur_plan].copy()
df_questions = df_questions.reset_index(drop=True)

df_questions.insert(
    0, "Question", range(1, len(df_questions) + 1)
)

print("Nombre de questions :", len(df_questions))
print("D-error final :", meilleur_d_error)
print("Équilibre des utilités (%) :", meilleur_equilibre)

display(df_questions)

Départ 1/5 terminé — D-error : 0.2270
Départ 2/5 terminé — D-error : 0.2270
Départ 3/5 terminé — D-error : 0.2270
Départ 4/5 terminé — D-error : 0.2270
Départ 5/5 terminé — D-error : 0.2270
Nombre de questions : 12
D-error final : 0.2269882182432727
Équilibre des utilités (%) : 75.85366856275279


,Question,Prix_SM,Hygiene_SM,Tailles_SM,Retours_SM,Prix_neuf,Tailles_neuf,Retours_neuf
0,1,20,2,1,2,40,2,1
1,2,20,2,1,1,40,2,2
2,3,20,1,1,1,100,1,2
3,4,40,2,2,1,60,1,2
4,5,20,1,1,2,40,2,2
5,6,40,2,1,1,60,2,1
6,7,20,1,1,1,100,2,1
7,8,40,2,1,2,40,2,1
8,9,20,1,1,1,40,2,2
9,10,40,2,2,1,40,1,2


In [ ]:
# Dictionnaires de correspondance code -> libellé
labels_hygiene = {1: "Standard", 2: "Premium/certifié"}
labels_assortiment = {1: "Limité", 2: "Extensif"}
labels_garantie = {1: "Sans retour", 2: "Retours 30j"}

df_questions_lisible = df_questions.copy()

df_questions_lisible["Hygiene_SM"] = df_questions_lisible["Hygiene_SM"].replace(labels_hygiene)
df_questions_lisible["Tailles_SM"] = df_questions_lisible["Tailles_SM"].replace(labels_assortiment)
df_questions_lisible["Retours_SM"] = df_questions_lisible["Retours_SM"].replace(labels_garantie)
df_questions_lisible["Tailles_neuf"] = df_questions_lisible["Tailles_neuf"].replace(labels_assortiment)
df_questions_lisible["Retours_neuf"] = df_questions_lisible["Retours_neuf"].replace(labels_garantie)

display(df_questions_lisible)

,Question,Prix_SM,Hygiene_SM,Tailles_SM,Retours_SM,Prix_neuf,Tailles_neuf,Retours_neuf
0,1,20,Premium/certifié,Limité,Retours 30j,40,Extensif,Sans retour
1,2,20,Premium/certifié,Limité,Sans retour,40,Extensif,Retours 30j
2,3,20,Standard,Limité,Sans retour,100,Limité,Retours 30j
3,4,40,Premium/certifié,Extensif,Sans retour,60,Limité,Retours 30j
4,5,20,Standard,Limité,Retours 30j,40,Extensif,Retours 30j
5,6,40,Premium/certifié,Limité,Sans retour,60,Extensif,Sans retour
6,7,20,Standard,Limité,Sans retour,100,Extensif,Sans retour
7,8,40,Premium/certifié,Limité,Retours 30j,40,Extensif,Sans retour
8,9,20,Standard,Limité,Sans retour,40,Extensif,Retours 30j
9,10,40,Premium/certifié,Extensif,Sans retour,40,Limité,Retours 30j


**Choix de modélisation retenu**

- Prix linéaire : un seul coefficient par franc (beta_prix). Permet de calculer la disposition à payer (ex. : combien de CHF vaut l'hygiène certifiée).
- Retours séparés : beta_retours_sm et beta_retours_neuf. Hypothèse : la possibilité de retour rassure davantage pour un pull de seconde main (risque sur l'état, la propreté, la taille) que pour un pull neuf. Séparer les deux permet de tester cette hypothèse statistiquement.
- Tailles communes : un seul beta_tailles pour les deux alternatives (pas de raison claire de penser que l'effet diffère).
- Hygiène : seulement pour la seconde main (un pull neuf est forcément propre).
- Questions évidentes retirées : 63 paires candidates restantes, toutes avec un vrai arbitrage.
- Au moins 4 questions avec hygiène standard, pour pouvoir mesurer l'effet de l'hygiène.

**Conséquences**
- Avec un paramètre de plus, il faut un peu plus de répondants pour une même